## Library Import

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter, deque
from tqdm import tqdm
import random
import copy
from PIL import Image
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import time
from collections import defaultdict
from pprint import pprint


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
org_path="/mnt/Velocity_Vault/Project_Storage/raw_dataset/"

In [3]:
dataset_org_path = "/mnt/Extra/Project_Storage/stereo_ML_dataset/org/"
dataset_path = "/mnt/Extra/Project_Storage/stereo_ML_dataset/"

left_patch_memmap = "left_patch.dat"
left_feat_memmap = "left_feat.dat"
right_strip_memmap = "right_strip.dat"
right_feat_memmap = "right_feat.dat"
patch_disparity_memmap = "patch_disp.dat"

## PLotting Functions

In [4]:

def display_image_array(img_array):
    
    # print(img_array.shape)

    
    """Display a numpy image array (2D or 3D) without axes."""
    plt.figure()
    if len(img_array.shape) == 3:  # RGB image
        plt.imshow(img_array)
    else:  # Grayscale
        plt.imshow(img_array, cmap='gray')
    plt.axis('off')
    plt.show()
    

def display_image_array_bulk(img_list):
    
    # print(img_array.shape)

    for img_array in img_list:
    
        """Display a numpy image array (2D or 3D) without axes."""
        plt.figure()
        if len(img_array.shape) == 3:  # RGB image
            plt.imshow(img_array)
        else:  # Grayscale
            plt.imshow(img_array, cmap='gray')
        plt.axis('off')
        plt.show()

In [5]:


def resize_image_array(image_array, scale_factor):
    # Convert array to PIL Image
    if len(image_array.shape) == 2:
        # Grayscale image
        img = Image.fromarray(image_array)
    elif len(image_array.shape) == 3:
        # RGB/RGBA image
        img = Image.fromarray(image_array.astype('uint8'))
    else:
        raise ValueError("Input array must be 2D (grayscale) or 3D (color)")
    
    # Calculate new dimensions
    width, height = img.size
    new_width = int(width * scale_factor)
    new_height = int(height * scale_factor)
    
    # Resize using Lanczos resampling (high quality)
    resized_img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
    
    # Convert back to numpy array
    resized_array = np.array(resized_img)
    
    # Preserve original dtype for grayscale
    if len(image_array.shape) == 2:
        resized_array = resized_array.astype(image_array.dtype)
    
    return resized_array

In [6]:
def plot_disp_array(data_array,colourbar=False):
    
    # print(data_array.shape)
    
    plt.figure(figsize=(6, 6))
    img = plt.imshow(data_array, cmap='viridis')
    plt.axis('off')
    if colourbar:
        cbar = plt.colorbar(img, fraction=0.046, pad=0.04)
        cbar.set_label('Value Scale', rotation=270, labelpad=15)
    
    plt.tight_layout()
    plt.show()

In [7]:
def plot_viridis_matrix(matrix):
    # Visualize results
    plt.figure(figsize=(12, 5))
    plt.imshow(matrix, cmap='viridis')
    plt.title('Matrix')
    plt.colorbar()

In [8]:
def display_six_arrays_with_cbar(img1, img2, img3, img4, disp1, disp2, 
                                show_colorbar=False, figsize=(15, 10)):
    """
    Display 6 arrays in a 3x2 grid layout with optional colorbars for disparity maps.
    
    Parameters:
    img1, img2, img3, img4: Image arrays (2D or 3D RGB)
    disp1, disp2: Disparity arrays (2D, displayed with viridis colormap)
    show_colorbar: Whether to show colorbars for disparity maps
    figsize: Tuple specifying the figure size (width, height)
    """
    if show_colorbar:
        fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    else:
        fig, axes = plt.subplots(3, 2, figsize=figsize)
    
    # First row: img1 and img2
    if len(img1.shape) == 3:
        axes[0, 0].imshow(img1)
    else:
        axes[0, 0].imshow(img1, cmap='gray')
    axes[0, 0].axis('off')
    
    if len(img2.shape) == 3:
        axes[0, 1].imshow(img2)
    else:
        axes[0, 1].imshow(img2, cmap='gray')
    axes[0, 1].axis('off')
    
    # Second row: img3 and img4
    if len(img3.shape) == 3:
        axes[1, 0].imshow(img3)
    else:
        axes[1, 0].imshow(img3, cmap='gray')
    axes[1, 0].axis('off')
    
    if len(img4.shape) == 3:
        axes[1, 1].imshow(img4)
    else:
        axes[1, 1].imshow(img4, cmap='gray')
    axes[1, 1].axis('off')
    
    # Third row: disp1 and disp2
    im1 = axes[2, 0].imshow(disp1, cmap='viridis')
    axes[2, 0].axis('off')
    if show_colorbar:
        plt.colorbar(im1, ax=axes[2, 0], fraction=0.046, pad=0.04)
    
    im2 = axes[2, 1].imshow(disp2, cmap='viridis')
    axes[2, 1].axis('off')
    if show_colorbar:
        plt.colorbar(im2, ax=axes[2, 1], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

## Loading Data

In [ ]:
import pickle

def load_variable(file_location):
    with open(file_location, 'rb') as f:
        return pickle.load(f)
    
variable_org_path = "/mnt/Velocity_Vault/Project_Storage/stereo_ML_dataset/raw_images/"

calib_list = load_variable(variable_org_path+"calib.pkl")
gray_image_list = load_variable(variable_org_path+"image.pkl")
truth_image_list = load_variable(variable_org_path+"truth.pkl")


resize_fraction = 1
resize_factor = 1/resize_fraction


maximum_disparity = int((700)*resize_factor)

# ......................

# calib_files = []
# truth_files = [{'org':[],'flip':[]}]
# image_files = [[{'org':[],'flip':[]}]]

# {"index":i,'left':truth_l,'right':truth_r}

#............................

## Patch Generation

In [ ]:

import cv2
import numpy as np

def compute_features(patch):
    """
    Compute 5 basic features from a normalized patch (values in [0,1]),
    and rescale each feature to [0,1] for better network input balance.
    """
    patch = patch.astype('float32')

    # 1. Basic stats
    mean = patch.mean()
    std = patch.std()

    # 2. Gradients
    grad_x = cv2.Sobel(patch, cv2.CV_32F, 1, 0)
    grad_y = cv2.Sobel(patch, cv2.CV_32F, 0, 1)
    sobel_x_mean = grad_x.mean()
    sobel_y_mean = grad_y.mean()

    # 3. Laplacian
    lap = cv2.Laplacian(patch, cv2.CV_32F)
    lap_mean = lap.mean()

    # -------------------------------
    # Normalize each feature to [0,1]
    # -------------------------------
    mean_n  = mean                           # already [0,1]
    std_n   = std * 2.0                    # since max std ≈ 0.5
    sobel_x_n = (sobel_x_mean + 1) / 2       # map [-1,1] → [0,1]
    sobel_y_n = (sobel_y_mean + 1) / 2       # map [-1,1] → [0,1]
    lap_n     = (lap_mean + 1) / 2           # map [-1,1] → [0,1]

    # Clip to valid range
    features = np.clip(
        np.array([mean_n, std_n, sobel_x_n, sobel_y_n, lap_n], dtype='float16'),
        0.0, 1.0
    )

    return features

def feature_error(feat1,feat2):
    
    diff = np.abs(feat1-feat2)
    diff_bool = np.zeros_like(diff,dtype=np.bool_)
    
    diff_bool[0] = diff[0]>0.01
    diff_bool[1] = diff[1]>0.02
    diff_bool[2] = diff[2]>0.02
    diff_bool[3] = diff[3]>0.02
    diff_bool[4] = diff[4]>0.03
    
    final = diff_bool[0] or diff_bool[1] or diff_bool[2] or diff_bool[3] or diff_bool[4] 

    return final    


def generate_patches_strips(main_path, patch_shape, target, max_disp=100, maximum_tries=1000,flush_interval=10000,feat_dim=13,disp_skip=2):
    
    eff_max_disp = max_disp//(disp_skip+1)
    
    patch_list = np.memmap(main_path+left_patch_memmap, dtype=np.float16, mode='w+', shape=(target, patch_shape[0], patch_shape[1]))
    patch_feat =np.memmap(main_path+left_feat_memmap, dtype=np.float16, mode='w+', shape=(target, feat_dim))
    strip_list = np.memmap(main_path+right_strip_memmap, dtype=np.float16, mode='w+', shape=(target, eff_max_disp, patch_shape[0], patch_shape[1]))
    strip_feat =np.memmap(main_path+right_feat_memmap, dtype=np.float16, mode='w+', shape=(target,  eff_max_disp, feat_dim))
    disp_list = np.memmap(main_path+patch_disparity_memmap, dtype=np.int16, mode='w+', shape=(target,))
    
    
    pbar = tqdm(range(target), desc="Generating patches")
    
    for i in pbar:
        
        try_count = 0
        patch_found = False
        
        choices = ['org','flip']
        side = np.random.choice(choices)
        
        truth_random = np.random.choice(truth_image_list[side])
        truth_index=truth_random['index']
        image_random = np.random.choice(gray_image_list[side][truth_index])
        
        while try_count < maximum_tries and not patch_found:
            
            disp_patch = truth_random['left']
            patch_image = image_random['left']
            strip_image = image_random['right']
            
            image_height, image_width = disp_patch.shape
            
            y_min = np.random.randint(0, image_height - patch_shape[0])
            x_min = np.random.randint(max_disp, image_width - patch_shape[1])
            y_max = y_min + patch_shape[0]
            x_max = x_min + patch_shape[1]
            
            disp_mask = disp_patch[y_min:y_max, x_min:x_max]
            truth_disp = np.median(disp_mask)
            
            
            
            if 0 < truth_disp < max_disp:
                
                image_patch = patch_image[y_min:y_max, x_min:x_max].astype('float16') / 255.0
                image_feat = compute_features(image_patch)
                
                image_strip = np.empty(shape=(eff_max_disp,patch_shape[0],patch_shape[1]),dtype=np.float16)
                
                feat_disp = np.empty(shape=(eff_max_disp,feat_dim),dtype=np.float16)
                
                check_strip = strip_image[y_min:y_max, (x_min-int((truth_disp//3)*3)):(x_max-int((truth_disp//3)*3))].astype('float16') / 255.0
                check_feat = compute_features(check_strip)
                
                if not feature_error(image_feat, check_feat):
            
                    for temp_disp in range(eff_max_disp):
                        
                        eff_disp = int(temp_disp*(disp_skip+1))
                        
                        image_strip[temp_disp] = strip_image[y_min:y_max, (x_min-eff_disp):(x_max-eff_disp)].astype('float16') / 255.0
                        feat_disp[temp_disp] = compute_features(image_strip[temp_disp])
                    
                    patch_list[i] = image_patch
                    patch_feat[i] = image_feat
                    
                    strip_list[i] = image_strip
                    strip_feat[i] = feat_disp
                    
                    disp_list[i] = truth_disp//(disp_skip+1)
                    
                    patch_found = True
            
            try_count += 1
        
        if not patch_found:
            print(f"Warning: Could not find suitable patch for index {i} after {maximum_tries} tries")
            
            patch_list[i] = np.zeros(patch_shape, dtype=np.uint8)
            disp_list[i] = 0
            
            break
        
        if (i + 1) % flush_interval == 0:
            patch_list.flush()
            patch_feat.flush()
            disp_list.flush()
            strip_list.flush()
            strip_feat.flush()
    
    patch_list.flush()
    patch_feat.flush()
    disp_list.flush()
    strip_list.flush()
    strip_feat.flush()
    
    return patch_list, patch_feat, strip_list, strip_feat, disp_list



maximum_disparity = int((720)*resize_factor)

# global_patch_shape = (min(global_image_shape//30),min(global_image_shape//30))
global_patch_shape = (16,16)

feat_dim = 5

d_skip = 1

sample_target = int((2**17+2)*(10/8))
# sample_target=100


In [ ]:
train_path = dataset_path+"train/"
test_path = dataset_path+"test/"

train_split=int(0.8*sample_target)
test_split=int(0.2*sample_target)

patch_list, patch_feat, strip_list, strip_feat, disp_list = generate_patches_strips(train_path,global_patch_shape,target=train_split,max_disp=maximum_disparity,feat_dim=feat_dim,disp_skip=d_skip,maximum_tries=1000000)
# patch_list, disp_list, strip_list = generate_patches_strips(test_path,global_patch_shape,target=test_split,max_disp=maximum_disparity,maximum_tries=100000)


Generating patches:   0%|          | 120/131073 [00:01<29:35, 73.77it/s]

In [ ]:
# for i in range(10):
    
#     display_image_array(patch_list[i])
#     # display_image_array(strip_list[i])
#     print(disp_list[i])

In [ ]:

print(patch_list.shape)
print(patch_feat.shape)
print(strip_list.shape)
print(patch_list.shape)
print(strip_feat.shape)
print(maximum_disparity)
print(maximum_disparity//(d_skip+1))

for i in range(0,0):
    
    disp = disp_list[i]
    
    image_array = patch_list[i]
    strip_arr = strip_list[i][disp]
    display_image_array(image_array*255)
    display_image_array(strip_arr*255)
    
    print(patch_feat[i])
    print(strip_feat[i][disp])
    print("Error : ",np.mean(np.abs(strip_feat[i][disp]-patch_feat[i])))
    print(disp*(d_skip+1))
    
print("\n\n")
    
print(np.median(patch_feat))
print(np.max(patch_feat))
print(np.median(strip_feat))
print(np.max(strip_feat))
    

(131073, 16, 16)
(131073, 5)
(131073, 360, 16, 16)
(131073, 16, 16)
(131073, 360, 5)
720
360



0.0
0.9883
0.0
0.9883


In [ ]:
print(np.max(disp_list))

319


In [ ]:
dataset_detail = []

dataset_detail=[global_patch_shape[0],global_patch_shape[1],sample_target,maximum_disparity,resize_fraction,feat_dim,d_skip]

dataset_detail=np.array(dataset_detail)

np.save(dataset_org_path+"detail.npy",dataset_detail, allow_pickle=True)
